In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
M_regular_results = pd.read_csv("MRegularSeasonDetailedResults.csv")
M_tourney_results = pd.read_csv("MNCAATourneyDetailedResults.csv")
M_seeds = pd.read_csv("MNCAATourneySeeds.csv")

In [ ]:
CUTOFF_YEAR = 2006

regular_results = M_regular_results.loc[M_regular_results["Season"] >= CUTOFF_YEAR]
tourney_results = M_tourney_results.loc[M_tourney_results["Season"] >= CUTOFF_YEAR]
seeds = M_seeds.loc[M_seeds["Season"] >= CUTOFF_YEAR]

#### Meaning of each columns in MRegularSeasonDetailedResults.csv:
* Season: the season year. 2001 means the tournament was played in the 2000/01 season
* DayNum: day index within the season. smaller - earlier games, larger = later games.
* WTeamID: winning team ID
* WScore: points scored by the winner
* LTeamID: losing team ID
* LScore: points scored by loser
* WLoc: location from winner's perspective: H means home, A means away, N means neutral
* NumOT: number of overtime periods. 0 means normal, no OT
* WFGM: field goals made, both 2 and 3pt
* WFGA: field goals attempted, both 2 and 3 pt
* WFGM3: 3 pointers made
* WFGA3: 3 pointers attempted
* WFTM: free throws made
* WFTA: free throws attempted
* WOR: offensive rebounds (own missed shots)
* WDR: defensive rebounds (opponent missed shots)
* total rebounds = WOR + WDR
* WAst: assists
* WTO: number of turnovers
* WStl: number of steals
* WBlk: number of blocks
* WPF: number of personal fouls
* The rest of the columns are the same, just L instead of W meaning for the losing team instead.

#### Meaning of each column in MNCAATourneyDetailResults.csv:
* Season, daynum, WteamID, WScore, LTeamID, Lscore, WLoc, NumOT, all W... stats and L... stats same as above.

#### Meaning of each column in MNCAATourneySeeds.csv:
* Season: tournamet year
* Seed: strings like W01, X08, W16a. First letter is the region, next 2 digits is the seed number (01 is best, 16 is worst team), optional a/b suffix means play-in teams

### Perform EDA

In [8]:
regular_results.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
13862,2006,8,1165,75,1384,54,N,0,24,54,...,20,11,17,11,19,11,12,5,4,18
13863,2006,8,1393,68,1126,37,H,0,21,44,...,13,4,9,12,19,10,23,13,1,24
13864,2006,9,1107,90,1324,73,N,0,35,62,...,25,7,12,18,16,13,18,11,7,15
13865,2006,9,1126,63,1384,52,N,1,22,59,...,26,12,19,20,35,11,17,3,4,16
13866,2006,9,1196,80,1389,51,H,0,29,54,...,18,3,8,16,22,10,17,4,0,20


In [6]:
tourney_results.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,WFGM,WFGA,...,LFGA3,LFTM,LFTA,LOR,LDR,LAst,LTO,LStl,LBlk,LPF
192,2006,134,1284,71,1214,49,N,0,27,61,...,15,7,13,15,27,7,13,4,4,11
193,2006,136,1104,90,1266,85,N,0,28,55,...,22,14,19,13,15,13,12,9,4,22
194,2006,136,1130,88,1334,76,N,2,29,57,...,25,14,18,14,23,18,15,4,2,22
195,2006,136,1181,70,1380,54,N,0,24,49,...,13,6,10,16,16,8,16,14,4,20
196,2006,136,1196,76,1375,50,N,0,29,59,...,20,3,6,7,24,8,16,7,3,17


In [7]:
seeds.head()

,Season,Seed,TeamID
1349,2006,W01,1181
1350,2006,W02,1400
1351,2006,W03,1234
1352,2006,W04,1261
1353,2006,W05,1393


## Prepare and clean data

Our current dataset has stats for the W teamID and L teamID, which indirect exposes who was the winning and losing team. This is supposed to be kept a secret and is what we have to decide on. 

Thus, we need to reformulate the columns to hide winning and losing information. we can change it by creating teamA/teamB rows, and a column called Y that says 1 means team A won, 0 means team B won. we will then have to also shuffle the rows if not it would look like either all of team A and all of team B wins all the time because they were directly taken from WTeamID and LTeamID.

In [11]:
games_a = pd.DataFrame({
    "Season": tourney_results["Season"],
    "DayNum": tourney_results["DayNum"],
    "TeamA": tourney_results["WTeamID"],
    "TeamB": tourney_results["LTeamID"],
    "Y": 1
})

games_b = pd.DataFrame({
    "Season": tourney_results["Season"],
    "DayNum": tourney_results["DayNum"],
    "TeamA": tourney_results["LTeamID"],
    "TeamB": tourney_results["WTeamID"],
    "Y": 0
})

# Combine both
matchups = pd.concat([games_a, games_b], ignore_index=True)

# Shuffle rows
matchups = matchups.sample(frac=1, random_state=42).reset_index(drop=True)


print(matchups.head())
print(matchups["Y"].value_counts())


   Season  DayNum  TeamA  TeamB  Y
0    2009     137   1199   1458  0
1    2021     146   1166   1211  0
2    2018     137   1308   1155  0
3    2018     136   1460   1397  0
4    2023     139   1211   1395  1
Y
0    1190
1    1190
Name: count, dtype: int64
